# Statement (P&L) Tear Sheet

Build a quarterly P&L model, evaluate it, and render a `statement_tearsheet` — a
single-page income-statement summary (line-item matrix, revenue/EBITDA trend,
margins) with an optional variance-vs-plan section.

In [ ]:
import datetime as dt
import json

from finstack_quant import reporting
from finstack_quant.statements import Evaluator, ModelBuilder
from finstack_quant.statements_analytics import run_variance


def build_pl(model_id, revenue, cogs, opex):
    qs = ["2024Q1", "2024Q2", "2024Q3", "2024Q4", "2025Q1", "2025Q2", "2025Q3", "2025Q4"]
    b = ModelBuilder(model_id)
    b.periods("2024Q1..2025Q4", "2024Q4")
    b.value("revenue", list(zip(qs, revenue)))
    b.value("cogs", list(zip(qs, cogs)))
    b.compute("gross_profit", "revenue - cogs")
    b.value("opex", list(zip(qs, opex)))
    b.compute("ebitda", "gross_profit - opex")
    b.compute("net_income", "ebitda * 0.7")
    b.compute("gross_margin", "gross_profit / revenue * 100")
    b.compute("ebitda_margin", "ebitda / revenue * 100")
    return b.build()


base = build_pl(
    "acme",
    revenue=[100, 104, 108, 112, 116, 120, 124, 128],
    cogs=[55, 57, 59, 61, 63, 65, 67, 69],
    opex=[18, 18, 19, 19, 20, 20, 21, 21],
)
result = Evaluator().evaluate(base)
print("EBITDA 2025Q4:", result.get("ebitda", "2025Q4"))

## Full tear sheet

The tear sheet renders inline in Jupyter (its `_repr_html_`). Pass a fixed
`generated` date for reproducible output.

In [ ]:
reporting.statement_tearsheet(result, title="Acme Corp — P&L", generated=dt.date(2026, 6, 22))

## Variance vs an alternate plan

`run_variance` compares two evaluated models; pass the parsed result to the
sheet's variance section.

In [ ]:
plan = build_pl(
    "acme-plan",
    revenue=[100, 104, 108, 112, 120, 126, 132, 138],
    cogs=[55, 57, 59, 61, 64, 67, 70, 73],
    opex=[18, 18, 19, 19, 21, 22, 23, 24],
)
plan_result = Evaluator().evaluate(plan)
variance_config = json.dumps({
    "baseline_label": "base",
    "comparison_label": "plan",
    "metrics": ["revenue", "ebitda"],
    "periods": ["2025Q1", "2025Q2", "2025Q3", "2025Q4"],
})
variance = json.loads(run_variance(result.to_json(), plan_result.to_json(), variance_config))
reporting.statement_tearsheet(
    result, variance=variance, sections=["summary", "variance"], generated=dt.date(2026, 6, 22)
)

## Saving a standalone HTML file

```python
ts = reporting.statement_tearsheet(result, generated=dt.date(2026, 6, 22))
ts.save("statement_tearsheet.html")
```